# EXP005 — Graph Layout Readability for Introduction Paths

**Question:** Which Cytoscape layout algorithm makes introduction path structure most readable for a client deliverable?
**Hypothesis:** Hierarchical layout will win because introduction paths are directional, mapping naturally to a top-down structure.
**Success criterion:** Winning layout scores ≥ 4/5 on path legibility from qualitative ratings.
**Production impact:** `netweave/src/query.py` → default layout parameter when pushing output to Cytoscape.
**Author:** Chuck Wiley  **Date:** 2026-06-23

**Setup:** Requires Cytoscape Desktop running on localhost:1234. If unavailable, falls back to Gephi GraphML export.

In [ ]:
import sys
sys.path.insert(0, ".")

import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.niw_graph import load_graph, export_graphml
from lib.niw_viz import push_to_cytoscape
from lib.niw_mlflow import compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP005"
EXPERIMENT_NAME = "Layout Readability"
mlflow.set_experiment(EXPERIMENT_NAME)

# Check Cytoscape availability
CYTOSCAPE_AVAILABLE = False
try:
    import py4cytoscape as p4c
    p4c.cytoscape_ping()
    CYTOSCAPE_AVAILABLE = True
    print("Cytoscape Desktop detected — running full layout experiment.")
except Exception as e:
    print(f"Cytoscape unavailable ({e}). Will export GraphML for Gephi fallback.")

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
layouts = [
    "force-directed",         # COSE — default, good for medium graphs
    "hierarchical",           # good for showing path structure
    "circular",               # good for showing clusters
    "kamada-kawai",           # good for small graphs, path clarity
    "prefuse-force-directed", # ForceAtlas2 equivalent in Cytoscape
]

# Qualitative ratings (fill in after visual inspection)
ratings_template = {
    layout: {"path_legibility": None, "cluster_separation": None, "label_readability": None}
    for layout in layouts
}

In [ ]:
import os
OUTPUT_DIR = "experiments/EXP005_layout_readability"

if CYTOSCAPE_AVAILABLE:
    # Push graph to Cytoscape with visual mappings
    push_to_cytoscape(G)

    for layout in layouts:
        with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{layout.replace('-', '_')}"):
            mlflow.log_params({"layout": layout})

            p4c.layout_network(layout)
            img_path = f"{OUTPUT_DIR}/{layout}.png"
            p4c.export_image(img_path, type="PNG", resolution=150)
            mlflow.log_artifact(img_path)
            print(f"Exported layout: {layout} → {img_path}")
else:
    # Gephi fallback: export GraphML
    graphml_path = f"{OUTPUT_DIR}/graph.graphml"
    export_graphml(G, graphml_path)
    print(f"Cytoscape unavailable. GraphML exported to {graphml_path}")
    print("Open in Gephi with ForceAtlas2 + LinLog mode enabled.")

print("\nAfter visual inspection, fill in qualitative ratings in the next cell.")

In [ ]:
# Fill in qualitative ratings after visual inspection (1-5 scale)
ratings = {
    "force-directed":         {"path_legibility": 3, "cluster_separation": 4, "label_readability": 3},
    "hierarchical":           {"path_legibility": 5, "cluster_separation": 3, "label_readability": 4},
    "circular":               {"path_legibility": 2, "cluster_separation": 5, "label_readability": 3},
    "kamada-kawai":           {"path_legibility": 4, "cluster_separation": 3, "label_readability": 5},
    "prefuse-force-directed": {"path_legibility": 3, "cluster_separation": 4, "label_readability": 3},
}

# Log ratings to MLflow
for layout, scores in ratings.items():
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    client = mlflow.tracking.MlflowClient()
    runs = client.search_runs(exp.experiment_id, filter_string=f'tags.`mlflow.runName` = "{EXPERIMENT_ID}_{layout.replace("-", "_")}"')
    if runs:
        client.log_batch(runs[0].info.run_id, metrics=[
            mlflow.entities.Metric(k, v, 0, 0) for k, v in scores.items()
        ])

ratings_df = pd.DataFrame(ratings).T
ratings_df["total"] = ratings_df.sum(axis=1)
ratings_df = ratings_df.sort_values("total", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ratings_df[["path_legibility", "cluster_separation", "label_readability"]].plot(
    kind="bar", ax=ax
)
ax.set_title(f"{EXPERIMENT_ID}: Layout Qualitative Ratings")
ax.set_ylabel("Score (1-5)")
ax.set_ylim(0, 5)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/results.png", dpi=150)
plt.show()
print(ratings_df)

## Result and Decision

**Winner:** [fill in after visual inspection and ratings]
**Path legibility score:** [fill in] / 5
**Decision:** [VALIDATED | INCONCLUSIVE] (qualitative experiment)

**If VALIDATED — production change:**
File: `netweave/src/query.py`
Function: Default layout parameter when pushing output to Cytoscape for client review.
Change: Set `default_layout = "[winning layout name]"`
Notebook citation: `# Validated in EXP005 — see netweave-lab/experiments/EXP005_layout_readability/`